# 贵州茅台 2024 年报 RAG 问答系统

基于 143 页中文年报的检索增强问答流水线，支持口语提问、带页码引用的回答。

**流水线**
```
PDF 解析 → 页眉清洗 → 父子块切分 → 千问向量化 → Chroma 索引
        → 查询改写 → BM25 + 向量混合召回 → bge-reranker 精排
        → 父块回填 → DeepSeek 生成 → RAGAS 四指标评测
```

**技术栈**：LangChain / Chroma / BM25(jieba) / bge-reranker-base / 通义千问 text-embedding-v4 / DeepSeek / RAGAS

**运行前提**
1. 环境变量：`.env` 中配置 `DASHSCOPE_API_KEY`（嵌入）与 `DEEPSEEK_API_KEY`（生成与判官）
2. 按顺序从上到下执行；`chroma_db_v2` 与 `parents_store` 必须来自同一次运行，不可混用
3. 仅重建索引时运行「建库」cell，日常提问无需重建


## 0. 环境依赖

In [ ]:
%pip install -q rank-bm25 jieba sentence-transformers modelscope
# 首次运行需要；已有环境可跳过。其余核心依赖见 requirements.txt


## 1. 模型初始化

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
)

print(model.invoke("用一句话介绍你自己").content)

## 2. 文档加载与预处理

### 2.1 加载 PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

docs = PyPDFLoader("贵州茅台2024年年度报告.pdf").load()

print("页数:", len(docs))
print("metadata:", docs[0].metadata)
print("前 300 字:", docs[0].page_content[:300])
# 第 5 页含主要会计数据表；`page` 从 0 开始，因此 page=5 是第 6 页

print(docs[5].page_content)

### 2.2 页眉清洗 + 父子块切分

每逢新页顶部都有「贵州茅台酒股份有限公司 2024 年年度报告」与「页码/总页数」两行，混入正文会严重污染检索，先按正则清除再切分。

- 父块 1500 字、**零重叠**（不参与检索，仅作为回填上下文，重叠只会重复占用 token）
- 子块 300 字、重叠 50（参与检索，需防止语义被切断）
- 每个子块写入 `parent_id`，建立子块 → 父块的映射


In [ ]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# ---------- 1. 页眉清理函数 ----------
hdr = re.compile(
    r"^\s*贵州茅台酒股份有限公司\s*\d*\s*年年度报告\s*$\s*\n?\s*\d+\s*/\s*\d+\s*$\s*",
    re.MULTILINE,
)

def strip_header(text: str) -> str:
    return hdr.sub("", text).strip()

# ---------- 2. 先清 docs 本身 ----------
docs_clean = [
    Document(page_content=strip_header(d.page_content), metadata=d.metadata)
    for d in docs
]

# ---------- 3. 父子切分器 ----------
SEPS = ["\n\n", "\n", "。", "！", "？", "；", "，", " ", ""]

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, chunk_overlap=0, separators=SEPS)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50, separators=SEPS)

# ---------- 4. 父块 + 映射 ----------
parents = parent_splitter.split_documents(docs_clean)

parents_store = {}   # parent_id -> 父块文本
children = []        # 子块列表

for p_idx, p_doc in enumerate(parents):
    pid = f"p{p_idx}"
    parents_store[pid] = p_doc.page_content

    for c_doc in child_splitter.split_documents([p_doc]):
        children.append(Document(
            page_content=c_doc.page_content,
            metadata={
                **c_doc.metadata,
                "parent_id": pid,
                "chunk_id": f"{pid}_c{len(children)}",
            },
        ))

print(f"父块 {len(parents_store)} 个，子块 {len(children)} 个")

## 3. 向量化与索引

### 3.1 嵌入模型（通义千问 text-embedding-v4，1024 维）

`check_embedding_ctx_length=False` 必须加：LangChain 默认会先用 tiktoken 编码成 token 数组，
而 DashScope 兼容接口只接受字符串，不关会报 `contents is neither str nor list of str`。


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

embeddings = OpenAIEmbeddings(
    model="text-embedding-v4",
    dimensions=1024,
    chunk_size=10,                        # DashScope 单次上限 10 条
    check_embedding_ctx_length=False,     # ← 关键：不加会报 contents is neither str nor list of str
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

### 3.2 建库

> **注意**：以下 cell 会先删除 `./chroma_db_v2` 再全量重建，会调用 72 次嵌入接口。
> 只在首次建库或需要重建时运行；日常提问直接跳过，改用 `Chroma(...)` 加载已有库。


In [ ]:
import shutil, os

if os.path.exists("./chroma_db_v2"):
    shutil.rmtree("./chroma_db_v2")

vs_clean = Chroma.from_documents(
    documents=children,
    embedding=embeddings,
    persist_directory="./chroma_db_v2",
    collection_name="maotai_v4_v2",
)
print("条数 =", vs_clean._collection.count())

In [ ]:
# 已有库时用这一格加载（不重新计算向量）
# vs_clean = Chroma(
#     persist_directory="./chroma_db_v2",
#     embedding_function=embeddings,
#     collection_name="maotai_v4_v2",
# )
# print("库内条数:", vs_clean._collection.count())   # 应为 717

## 4. 检索链路

### 4.1 查询改写

把口语问题改写成能命中财务报告表述的查询：补全实体、对齐术语、保留疑问焦点。
开启后 Context Recall 从 0.87 提升至 0.97，但会引入同指标不同口径的片段，需配合生成侧的口径约束。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

REPORT_YEAR = 2024          # 年报年度，明确写死

rewrite_prompt = ChatPromptTemplate.from_template(
    "你是企业年报知识库的检索查询改写助手。\n"
    "知识库内容为贵州茅台2024年年度报告。\n\n"
    "【时间归一】\n"
    "用户说“去年”“今年”“本年度”“前年”等相对时间时，一律按 {RY} 年计算，"
    "不要按当前日期推算。\n\n"
    "【改写要求】\n"
    "1. 补全实体：“茅台”补成“贵州茅台”，“年报/报告/公司”理解为“{RY}年年度报告”。\n"
    "2. 术语对齐：把口语替换成年报中的标准表述，例如\n"
    "   赚了多少钱/净利润 → 归属于上市公司股东的净利润\n"
    "   营收/收入 → 营业收入\n"
    "   每股赚多少 → 基本每股收益\n"
    "   分红/派息 → 现金红利（每10股）\n"
    "   负债水平 → 资产负债率\n"
    "3. 保留疑问焦点：写清问的是“是多少”“有哪些”“是否”“占比多少”中的哪一种，"
    "不要把问句缩成名词堆砌。\n"
    "4. 数字、年份、专有名词原样保留，不增不删不换算。\n"
    "5. 控制在30字以内，不要堆砌同义词。\n\n"
    "【约束】\n"
    "- 只输出改写结果一行，不要解释，不要加引号或“改写后：”之类的前缀\n"
    "- 若问题涉及年报通常不披露的内容（如个人联系方式、未披露的细分数据），"
    "原样保留问题，不要强行改写成看似合理的财务指标\n"
    "- 不要臆造年报中不存在的指标名称\n\n"
    "问题：{q}\n"
    "改写结果："
)


rewrite_chain = rewrite_prompt | model | StrOutputParser()

question = "茅台去年赚了多少钱？"
rewritten = rewrite_chain.invoke({"q": question, "RY": REPORT_YEAR})

print("原问题:", question)
print("改写后:", rewritten)

### 4.2 混合召回（BM25 + 向量）

`jieba.cut_for_search` 是搜索模式，会把长词再切一遍，保证「茅台」这类短词一定能被切出；
`preprocess_func` 在建索引与查询时都会调用，两者必须一致，否则 BM25 会全部 0 分且不报错。

`EnsembleRetriever` 采用 RRF（倒数排名融合），`weights` 作用于排名而非分数。


In [ ]:
import jieba
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
def zh_tokenize(text: str) -> list[str]:
    return [w for w in jieba.cut_for_search(text) if w.strip()]

bm25 = BM25Retriever.from_documents(children, k=10, preprocess_func=zh_tokenize)
vec = vs_clean.as_retriever(search_kwargs={"k": 10})
ensemble = EnsembleRetriever(retrievers=[bm25, vec], weights=[0.4, 0.6])

hits = ensemble.invoke("贵州茅台的核心竞争力是什么？")
for i, d in enumerate(hits[:3]):
    print(f"[{i}] p{d.metadata.get('page_label')} | {repr(d.page_content[:50])}")

### 4.3 精排（bge-reranker-base）

交叉编码器把「问题 + 子块」拼成一个序列送入 BERT，逐对打分，精度远高于双塔向量，
但无法预计算，只能在候选集上做精排。

HuggingFace 直连不稳定，走 ModelScope 下载。


In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download("BAAI/bge-reranker-base")
print("下载到:", model_dir)

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(model_dir, max_length=512)
print("reranker 就绪")

In [ ]:
import time

def rerank(question, docs, top_n=6):
    if not docs:
        return []
    pairs = [(question, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    order = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)
    return [(docs[i], float(scores[i])) for i in order[:top_n]]


q = "贵州茅台的核心竞争力是什么？"

print("=== 无精排 ===")
for i, d in enumerate(ensemble.invoke(q)[:6]):
    print(f"[{i}] p{d.metadata.get('page_label')} | {d.page_content[:70]}")

print("\n=== 有精排 ===")
t0 = time.time()
for i, (d, s) in enumerate(rerank(q, ensemble.invoke(q))):
    print(f"[{i}] p{d.metadata.get('page_label')} score={s:+.3f} | {d.page_content[:70]}")
print(f"\n耗时 {time.time()-t0:.2f}s")

### 4.4 上下文组装（父块回填）

精排后按 `parent_id` 去重（同一父块只取分数最高的那条子块），再回填父块全文。

- `fetch_k`：精排保留的子块数，**真正控制上下文规模的开关**
- `n_pages`：期望的父块数；子块扎堆时实际取到的可能少于此值


In [ ]:
USE_REWRITE = True   # 想要改写就改 True

def build_context(question, fetch_k=5, n_pages=6):
    q = rewrite_chain.invoke({"q": question, "RY": 2024}) if USE_REWRITE else question

    raw = ensemble.invoke(q)
    ranked = rerank(question, raw, top_n=fetch_k)

    seen_parent, picked = set(), []
    for d, score in ranked:
        pid = d.metadata["parent_id"]
        if pid in seen_parent:
            continue
        seen_parent.add(pid)
        picked.append((pid, d.metadata.get("page_label"), score))
        if len(picked) >= n_pages:
            break

    parts = [f"[来源: 第{page}页]\n{parents_store[pid]}" for pid, page, _ in picked]
    return "\n\n".join(parts), picked

## 5. 生成

提示词的核心约束：
1. 数字照抄原文，不换算、不自行计算（自算的派生数值在原文中找不到，会拉低 Faithfulness）
2. 资料没有的一律回答「资料中未提及」，禁止用常识填补
3. 同一指标多口径时逐一区分，优先合并报表本期数据
4. 回答完整成句并注明页码（篇幅过短会导致反推问题失效，影响 Answer Relevancy）


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "你是贵州茅台2024年年度报告的财报分析助手。你唯一的资料来源是下面提供的年报片段，"
     "除此之外没有任何其他信息来源，不要进行与片段无关的联想生成。\n\n"
     "【回答规则】\n"
     "1. 事实与数字只能来自资料原文，并照抄原文的数值、单位与口径。\n"
     "   不要换算（如把元改成亿元）、不要四舍五入、不要补全省略的位数；\n"
     "   也不要自行做加减乘除或推算增减额——凡原文未直接给出的派生数值，一律不要计算。\n"
     "2. 资料中没有明确写出的内容，一律回答「资料中未提及」。不要用行业常识、"
     "往年数据、公开新闻或逻辑推理来填补。宁可少答，不可猜测。\n"
     "3. 资料只覆盖部分内容时，明确说明哪部分有、哪部分没有，"
     "例如「资料提及了A，但未提及B」。\n"
     "4. 回答要完整成句，让不看检索过程的人也能读懂。先点明问的是哪个指标或事项，"
     "再给数值与同比变化。\n"
     "   示例：用户问「去年赚了多少钱」，应回答「您问的应是归属于上市公司股东的净利润，"
     "2024年为86,228,146,421.62元，同比增长15.38%」，而不是只回一个数字。\n"
     "5. 同一指标在资料中出现多个版本时（合并报表/母公司报表、本期/上期、"
     "含税/不含税、营业收入/营业总收入、净利润/归母净利润），必须逐一区分，"
     "只采用与问题相符的那一个，并在数值后注明口径，如「（合并报表口径）」。"
     "严禁把不同口径的数值混用、相加或相互比较。问题未指明口径时，"
     "默认采用合并报表本期数据。\n"
     "6. 不要整段复述资料原文，但也不要只甩一串数字。回答控制在350字以内，"
     "问什么答什么，不罗列资料里其他无关数据，不做趋势预测或投资建议。\n"
     "7. 回答末尾单独一行注明依据页码，格式「依据：第X页」；"
     "同一信息多页出现时只引最直接的那一页。资料未提及时写「依据：无」。\n"
     "8. 资料确实没有对应内容时，除回答「资料中未提及」外，"
     "补一句说明查的是哪项信息，例如「资料中未提及研发人员数量的披露」。\n\n"
     "【资料】\n{context}"),
    ("human", "{question}"),
])

rag_chain = rag_prompt | model | StrOutputParser()

In [ ]:
# 单条端到端验证
q = "贵州茅台的核心竞争力是什么？"
ctx, picked = build_context(q)

print("命中父块:", [(p, pg) for p, pg, _ in picked])
print("context 字数:", len(ctx))
print()

print(rag_chain.invoke({"context": ctx, "question": q}))

## 6. 评测（RAGAS 四指标）

| 指标 | 衡量什么 | 依赖 |
|---|---|---|
| Faithfulness | 答案的每句陈述能否在检索内容中找到依据 | llm |
| Answer Relevancy | 由答案反推问题与原问题的相似度 | llm + embeddings |
| Context Precision | 检索内容中有多少是相关的 | llm |
| Context Recall | 参考答案的要点有多少被检索到 | llm |

> 判官使用 `deepseek-chat`，与生成模型同源，存在 self-preference 偏差，分数偏乐观。
> 另：`ResponseRelevancy` 只存在于旧路径 `ragas.metrics`，`ragas.metrics.collections` 下的
> `AnswerRelevancy` 是 v2 架构（需 `llm_factory` + `ascore`），不能传给 `evaluate()`。


### 6.1 导入与兼容补丁

In [ ]:
import sys, types

# ragas 会导入 langchain_community.chat_models.vertexai，当前环境缺少该模块，塞一个空壳
mod = types.ModuleType("langchain_community.chat_models.vertexai")

class ChatVertexAI:
    pass

mod.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = mod

from datasets import Dataset
from ragas import evaluate as ragas_evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    ContextPrecision,
    ContextRecall,
)

print("导入成功")

### 6.2 评测集（20 条）

- `fact` 11 条：精确数值与事实
- `summary` 2 条：需要跨段归纳
- `colloquial` 2 条：口语化提问，检验查询改写
- `negative` 5 条：年报中不存在的信息，检验拒答能力


In [ ]:
CASES = [
    {"id": "F01", "type": "fact",
     "question": "2024年贵州茅台的营业收入是多少？",
     "reference": "2024年营业收入 170,899,152,276.34 元，上年 147,693,604,994.14 元，同比增长15.71%。",
     "expected_pages": ["5", "6", "8", "56"]},
    {"id": "F02", "type": "fact",
     "question": "2024年归属于上市公司股东的净利润是多少？",
     "reference": "归属于上市公司股东的净利润 86,228,146,421.62 元，同比增长15.38%。",
     "expected_pages": ["5", "6", "8", "56"]},
    {"id": "F03", "type": "fact",
     "question": "公司每10股派发现金红利多少元？",
     "reference": "每10股派发现金红利276.24元（含税），合计派发34,671,299,522.40元（含税）。",
     "expected_pages": ["2"]},
    {"id": "F04", "type": "fact",
     "question": "2024年基本每股收益是多少？",
     "reference": "基本每股收益 68.64 元/股，2023年为59.49元/股。",
     "expected_pages": ["5", "6"]},
    {"id": "F05", "type": "fact",
     "question": "2024年加权平均净资产收益率是多少？",
     "reference": "加权平均净资产收益率 36.02%，上年34.19%，增加1.83个百分点。",
     "expected_pages": ["5", "6"]},
    {"id": "F06", "type": "fact",
     "question": "经营活动产生的现金流量净额是多少？",
     "reference": "经营活动产生的现金流量净额 92,463,692,168.43 元，同比增长38.85%。",
     "expected_pages": ["5", "6", "8", "56"]},
    {"id": "F07", "type": "fact",
     "question": "公司的股票代码是多少？",
     "reference": "公司代码 600519，公司简称贵州茅台，上市于上海证券交易所。",
     "expected_pages": ["1"]},
    {"id": "F08", "type": "fact",
     "question": "为公司出具审计报告的会计师事务所是哪家？",
     "reference": "天健会计师事务所（特殊普通合伙），出具标准无保留意见审计报告。",
     "expected_pages": ["2", "5", "8"]},
    {"id": "F09", "type": "fact",
     "question": "2024年茅台酒和系列酒的营业收入分别是多少？",
     "reference": "资料按产品类型分别披露了茅台酒、系列酒的营业收入，其中系列酒营业收入约246.84亿元。",
     "expected_pages": ["15", "16", "17", "18", "19", "20", "21", "107", "141"]},
    {"id": "F10", "type": "fact",
     "question": "公司采用直接销售还是批发代理渠道？",
     "reference": "分为直销和批发代理两类渠道。",
     "expected_pages": ["8", "16", "17", "18", "19", "20", "21"]},
    {"id": "F11", "type": "fact",
     "question": "公司2024年的研发投入占营业收入的比例是多少？",
     "reference": "研发投入总额占营业收入比例为0.41%，研发投入合计695,376,735.81元。",
     "expected_pages": ["8", "9", "119"]},

    {"id": "S01", "type": "summary",
     "question": "贵州茅台的核心竞争力是什么？",
     "reference": "品质、品牌、工艺、文化、环境五大核心竞争力。",
     "expected_pages": ["20", "21"]},
    {"id": "S02", "type": "summary",
     "question": "公司的利润分配政策是什么？",
     "reference": "拟以股权登记日总股本扣除回购专户股份为基数派发现金红利，每10股派276.24元（含税）。",
     "expected_pages": ["2", "3", "4"]},

    {"id": "C01", "type": "colloquial",
     "question": "茅台去年赚了多少钱？",
     "reference": "如指2024年，归母净利润86,228,146,421.62元。",
     "expected_pages": ["5", "6", "8", "56"]},
    {"id": "C02", "type": "colloquial",
     "question": "给股东分了多少钱？",
     "reference": "拟派发现金红利合计34,671,299,522.40元（含税）。",
     "expected_pages": ["2"]},

    {"id": "N02", "type": "negative", "must_refuse": True,
     "question": "公司在美国市场的销售额是多少？",
     "reference": "资料未提及。年报仅按境内、境外披露营业收入。",
     "expected_pages": []},
    {"id": "N03", "type": "negative", "must_refuse": True,
     "question": "公司2025年第一季度的营业收入是多少？",
     "reference": "资料未提及。年报只覆盖2024年度。",
     "expected_pages": []},
    {"id": "N04", "type": "negative", "must_refuse": True,
     "question": "公司共有多少名研发人员？",
     "reference": "资料未提及研发人员数量。",
     "expected_pages": []},
    {"id": "N05", "type": "negative", "must_refuse": True,
     "question": "公司董事长的手机号是多少？",
     "reference": "资料未提及任何个人联系方式。",
     "expected_pages": []},
    {"id": "N06", "type": "negative", "must_refuse": True,
     "question": "茅台在日本市场的经销商有多少家？",
     "reference": "资料未提及日本市场经销商数量。",
     "expected_pages": []},
]

from collections import Counter
print(f"共 {len(CASES)} 条:", dict(Counter(c["type"] for c in CASES)))

### 6.3 攒样本

`case_id` / `case_type` 一并写入，RAGAS 会原样带进结果表，便于后续分组统计。


In [ ]:
import time
from datasets import Dataset

rows = []
for c in CASES:
    ctx, picked = build_context(c["question"])
    answer = rag_chain.invoke({"context": ctx, "question": c["question"]})
    rows.append({
        "user_input": c["question"],
        "reference": c["reference"],
        "response": answer,
        "retrieved_contexts": [parents_store[pid] for pid, _, _ in picked],
        "case_id": c["id"],
        "case_type": c["type"],
    })
    print(f"{c['id']} ✓ {len(answer)}字 / {len(picked)}块")

print(f"\n共 {len(rows)} 条")
dataset = Dataset.from_list(rows)

### 6.4 四指标评测

注意：`Faithfulness` / `ContextPrecision` / `ContextRecall` 只接受 `llm`；
多传 `embeddings` 会报 `unexpected keyword argument`。只有 `ResponseRelevancy` 需要嵌入。


In [ ]:
from langchain.chat_models import init_chat_model

judge_llm = init_chat_model("deepseek:deepseek-chat", temperature=0)   # 判官
emb = embeddings                                                      # 复用千问 v4

result = ragas_evaluate(
    dataset,
    metrics=[Faithfulness(llm=judge_llm),
             ResponseRelevancy(llm=judge_llm, embeddings=emb),
             ContextPrecision(llm=judge_llm),
             ContextRecall(llm=judge_llm)],
    llm=judge_llm,
    embeddings=emb,
)
print(result)

### 6.5 分组统计

拒答题在 Faithfulness 与 Answer Relevancy 下天然低分：正确拒答的陈述找不到原文依据，
短答案也无法反推出有效问题。因此按「可答类」与「拒答类」分开统计，再单独计算拒答准确率。


In [ ]:
import pandas as pd

cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

df = result.to_pandas()
df["case_id"]   = [c["id"] for c in CASES]
df["case_type"] = [c["type"] for c in CASES]

print("=== 20 条混算 ===")
print(df[cols].mean().round(4))

print("\n=== 按题型分组 ===")
print(df.groupby("case_type")[cols].mean().round(3))

print("\n=== 可答类（剔除拒答题）===")
print(df[df.case_type != "negative"][cols].mean().round(3))

REFUSE_KEYS = ["未提及", "没有提及", "未披露", "不包含", "无法", "未找到"]
neg = df[df.case_type == "negative"]
ok = neg.response.astype(str).apply(lambda s: any(k in s for k in REFUSE_KEYS))

print(f"\n=== 拒答准确率 {ok.sum()}/{len(neg)} = {ok.mean():.1%} ===")
for cid, passed, resp in zip(neg.case_id, ok, neg.response.astype(str)):
    print(f"{cid} {'✓' if passed else '✗'} | {resp[:60]}")

## 7. 交互提问

改 `q` 即可提问，无需重建索引。`build_context` 内部会依次执行「改写 → 混合召回 → 精排 → 父块回填」。


In [26]:
q = "报告期内对子公司的管理控制情况"

ctx, picked = build_context(q)
print("命中页码:", [pg for _, pg, _ in picked])
print(f"上下文 {len(ctx)} 字")
print("-" * 60)
print(rag_chain.invoke({"context": ctx, "question": q}))

命中页码: ['36', '33', '10', '74']
上下文 3863 字
------------------------------------------------------------
报告期内对子公司的管理控制情况：公司为加强对其子公司的管理、规范其议事机构及议事程序，不定期召开制度评审会，遵照管理的合法性原则、适用性原则、适时性原则、问题导向改进原则，前置评审子公司《章程》及“三会”议事规则，对其《章程》及“三会”议事规则的整体框架、职责职权及具体的议事范围进行研究、讨论；通过评审子公司章程及“三会”议事规则，强化公司对子公司领导班子权力运行的制约和监督，提高工作效率和工作水平，促进议事机构决策的合法化、制度化及科学化。资料未提及对子公司管控的其他具体量化指标。

依据：第36页


In [ ]:
# 需要看检索细节时用这一格
q = "公司的茅台酒设计产能是多少？"

print("改写后 :", rewrite_chain.invoke({"q": q, "RY": REPORT_YEAR}))

raw = ensemble.invoke(q)
print("召回条数:", len(raw))

for d, s in rerank(q, raw, top_n=5):
    print(f"  {s:+.2f}  p{d.metadata.get('page_label'):<4} | {d.page_content[:40]}")